In [ ]:


    import torch
    import torch.nn as nn
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from torch.utils.data import DataLoader, TensorDataset
    import joblib
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import os


    CURVE_LEN  = 1001
    EPOCHS     = 400
    BATCH      = 32
    LR         = 1e-4
    VAL_SPLIT  = 0.15
    PATIENCE   = 50
    SEED       = 42
    SAVE_DIR   = "."

    DATA_PATH  = r"C:\Users\satya\Desktop\Data\dataset_fullcurve.xlsx"

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")



    class InverseNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            )
            self.fc = nn.Sequential(
                nn.Linear(128 * (CURVE_LEN // 8), 256), nn.ReLU(),
                nn.Linear(256, 128), nn.ReLU(),
                nn.Linear(128, 3),
            )

        def forward(self, x):
            x = x.unsqueeze(1)
            x = self.conv(x)
            return self.fc(x.view(x.size(0), -1))


    class ForwardNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(3, 256),   nn.ReLU(),
                nn.Linear(256, 512), nn.ReLU(),
                nn.Linear(512, 1024), nn.ReLU(),
                nn.Linear(1024, 2048), nn.ReLU(),
                nn.Linear(2048, CURVE_LEN),
            )

        def forward(self, x):
            return self.net(x)


  
    print("Loading dataset...")
    df      = pd.read_excel(DATA_PATH)
    p_cols  = [f"P{i}" for i in range(CURVE_LEN)]

    X_curve  = np.nan_to_num(df[p_cols].values.astype(float))
    y_params = df[["L", "W", "G"]].values.astype(float)

    print(f"  {len(X_curve)} samples  |  "
        f"L [{y_params[:,0].min():.1f}–{y_params[:,0].max():.1f}]  "
        f"W [{y_params[:,1].min():.1f}–{y_params[:,1].max():.1f}]  "
        f"G [{y_params[:,2].min():.1f}–{y_params[:,2].max():.1f}]\n")


 
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_curve, y_params, test_size=VAL_SPLIT, random_state=SEED)



    sx_inv = StandardScaler().fit(X_tr)
    sy_inv = StandardScaler().fit(y_tr)
    sx_fwd = StandardScaler().fit(y_tr)
    sy_fwd = StandardScaler().fit(X_tr)

    def to_tensor(arr):
        return torch.tensor(arr, dtype=torch.float32)

   
    Xtr_inv  = to_tensor(sx_inv.transform(X_tr))
    ytr_inv  = to_tensor(sy_inv.transform(y_tr))
    Xval_inv = to_tensor(sx_inv.transform(X_val))
    yval_inv = to_tensor(sy_inv.transform(y_val))

   
    Xtr_fwd  = to_tensor(sx_fwd.transform(y_tr))
    ytr_fwd  = to_tensor(sy_fwd.transform(X_tr))
    Xval_fwd = to_tensor(sx_fwd.transform(y_val))
    yval_fwd = to_tensor(sy_fwd.transform(X_val))



    def make_dl(X, y, shuf=True):
        return DataLoader(TensorDataset(X, y), batch_size=BATCH, shuffle=shuf)

    dl_tr_inv  = make_dl(Xtr_inv,  ytr_inv)
    dl_val_inv = make_dl(Xval_inv, yval_inv, shuf=False)
    dl_tr_fwd  = make_dl(Xtr_fwd,  ytr_fwd)
    dl_val_fwd = make_dl(Xval_fwd, yval_fwd, shuf=False)


   
  
    inv_model = InverseNet().to(device)
    fwd_model = ForwardNet().to(device)

    opt_inv = torch.optim.Adam(inv_model.parameters(), lr=LR, weight_decay=1e-5)
    opt_fwd = torch.optim.Adam(fwd_model.parameters(), lr=LR, weight_decay=1e-5)

    sch_inv = torch.optim.lr_scheduler.CosineAnnealingLR(opt_inv, T_max=EPOCHS, eta_min=1e-6)
    sch_fwd = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fwd, T_max=EPOCHS, eta_min=1e-6)

    loss_fn = nn.MSELoss()


    def epoch_loss(model, loader, opt=None):
        model.train() if opt else model.eval()
        total = 0.0
        ctx = torch.enable_grad() if opt else torch.no_grad()
        with ctx:
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = loss_fn(pred, yb)
                if opt:
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                total += loss.item()
        return total / len(loader)


    def val_mae(model, loader):
        """Per-parameter MAE in original units."""
        model.eval()
        ps, ts = [], []
        with torch.no_grad():
            for xb, yb in loader:
                ps.append(model(xb.to(device)).cpu().numpy())
                ts.append(yb.numpy())
        ps = sy_inv.inverse_transform(np.vstack(ps))
        ts = sy_inv.inverse_transform(np.vstack(ts))
        return np.abs(ps - ts).mean(axis=0)



    print("=" * 60)
    print("  Training")
    print("=" * 60)

    hist = {k: [] for k in ["tr_inv","val_inv","tr_fwd","val_fwd","mL","mW","mG"]}
    best_inv = best_fwd = float("inf")
    patience_cnt = 0

    for ep in range(1, EPOCHS + 1):

        ti = epoch_loss(inv_model, dl_tr_inv, opt_inv)
        vi = epoch_loss(inv_model, dl_val_inv)
        tf = epoch_loss(fwd_model, dl_tr_fwd, opt_fwd)
        vf = epoch_loss(fwd_model, dl_val_fwd)
        sch_inv.step(); sch_fwd.step()

        mae = val_mae(inv_model, dl_val_inv)

        for k, v in zip(["tr_inv","val_inv","tr_fwd","val_fwd","mL","mW","mG"],
                        [ti, vi, tf, vf, *mae]):
            hist[k].append(v)

        print(f"Ep {ep:>3}/{EPOCHS}  INV {ti:.5f}/{vi:.5f}  "
            f"FWD {tf:.5f}/{vf:.5f}  "
            f"MAE L={mae[0]:.3f} W={mae[1]:.3f} G={mae[2]:.3f}")

        if vi < best_inv:
            best_inv = vi
            torch.save(inv_model.state_dict(), os.path.join(SAVE_DIR, "inverse_model.pth"))

        if vf < best_fwd:
            best_fwd = vf
            torch.save(fwd_model.state_dict(), os.path.join(SAVE_DIR, "forward_model.pth"))
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"\nEarly stop at epoch {ep}")
                break


    joblib.dump(sx_inv, os.path.join(SAVE_DIR, "sx_inv.pkl"))
    joblib.dump(sy_inv, os.path.join(SAVE_DIR, "sy_inv.pkl"))
    joblib.dump(sx_fwd, os.path.join(SAVE_DIR, "sx_fwd.pkl"))
    joblib.dump(sy_fwd, os.path.join(SAVE_DIR, "sy_fwd.pkl"))
    print("\n✅ Models and scalers saved.\n")



    ep_x = range(1, len(hist["tr_inv"]) + 1)


    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("Learning Curves", fontsize=13, fontweight="bold")

    for ax, tr_k, val_k, title in [
        (axes[0], "tr_inv", "val_inv", "InverseNet  (curve → L, W, G)"),
        (axes[1], "tr_fwd", "val_fwd", "ForwardNet  (L, W, G → curve)"),
    ]:
        ax.plot(ep_x, hist[tr_k],  label="Train",      color="#2563eb")
        ax.plot(ep_x, hist[val_k], label="Validation", color="#f59e0b", linestyle="--")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss (log)")
        ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "plot_loss_curves.png"), dpi=150)
    plt.close()
    print("Saved: plot_loss_curves.png")


    fig, ax = plt.subplots(figsize=(9, 4))
    ax.set_title("Validation MAE per Parameter  (InverseNet)", fontweight="bold")
    for k, lbl, col in [("mL","L","#2563eb"), ("mW","W","#10b981"), ("mG","G","#ef4444")]:
        ax.plot(ep_x, hist[k], label=lbl, color=col)
    ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (original units)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "plot_mae_per_param.png"), dpi=150)
    plt.close()
    print("Saved: plot_mae_per_param.png")

  
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(y_params[:, 2], bins=30, color="#6366f1", edgecolor="white", alpha=0.85)
    ax.axvline(0.36,  color="#ef4444", linestyle="--", linewidth=1.5, label="Test G min = 0.36")
    ax.axvline(23.7,  color="#ef4444", linestyle="--", linewidth=1.5, label="Test G max = 23.7")
    ax.axvline(y_params[:,2].min(), color="#10b981", linestyle=":", linewidth=1.2, label="Train G min")
    ax.axvline(y_params[:,2].max(), color="#10b981", linestyle=":", linewidth=1.2, label="Train G max")
    ax.set_title("G Distribution — Training Data vs Test Range", fontweight="bold")
    ax.set_xlabel("G"); ax.set_ylabel("Count")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "plot_G_distribution.png"), dpi=150)
    plt.close()
    print("Saved: plot_G_distribution.png")

    # Plot 4 — final epoch MAE bar chart
    final_mae = np.array([hist["mL"][-1], hist["mW"][-1], hist["mG"][-1]])
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ["#2563eb", "#10b981", "#ef4444"]
    bars = ax.bar(["L", "W", "G"], final_mae, color=colors, edgecolor="white", width=0.5)
    for bar, v in zip(bars, final_mae):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                ha="center", fontweight="bold", fontsize=10)
    ax.set_title("Final Validation MAE per Parameter", fontweight="bold")
    ax.set_ylabel("MAE (original units)"); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "plot_final_mae.png"), dpi=150)
    plt.close()
    print("Saved: plot_final_mae.png")

    print("\n✅ TRAINING COMPLETE")

Using device: cpu

Loading dataset...
  2789 samples  |  L [12.0–30.0]  W [1.0–8.0]  G [0.5–18.0]

  Training
Ep   1/400  INV 0.39355/0.16729  FWD 0.68492/0.47301  MAE L=0.492 W=0.584 G=2.416
Ep   2/400  INV 0.13343/0.10597  FWD 0.40441/0.38818  MAE L=0.497 W=0.430 G=1.838
Ep   3/400  INV 0.08342/0.07259  FWD 0.32861/0.31142  MAE L=0.280 W=0.348 G=1.488
Ep   4/400  INV 0.06410/0.06002  FWD 0.27037/0.27004  MAE L=0.250 W=0.297 G=1.491
Ep   5/400  INV 0.05103/0.05207  FWD 0.23686/0.24655  MAE L=0.385 W=0.287 G=1.208
Ep   6/400  INV 0.04460/0.04207  FWD 0.21458/0.22478  MAE L=0.313 W=0.310 G=0.980
Ep   7/400  INV 0.03947/0.03662  FWD 0.19866/0.21658  MAE L=0.346 W=0.255 G=0.913
Ep   8/400  INV 0.03392/0.03622  FWD 0.18317/0.19221  MAE L=0.370 W=0.238 G=0.861
Ep   9/400  INV 0.03176/0.03089  FWD 0.16852/0.18041  MAE L=0.247 W=0.222 G=0.877
Ep  10/400  INV 0.03011/0.03121  FWD 0.15812/0.17031  MAE L=0.297 W=0.239 G=0.827
Ep  11/400  INV 0.02806/0.02738  FWD 0.14856/0.16212  MAE L=0.267 W=0.

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from datetime import datetime


CURVE_LEN   = 1001
SAVE_DIR    = "."
TEST_PATH   = r"C:\Users\satya\Desktop\Data\testings.xlsx"
METRICS_FILE = "test_metrics_unclipped.txt"

N_RESTARTS = 3
ADAM_ITERS = 250
ADAM_LR    = 0.02
NELDER_ITERS = 300

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")


class InverseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * (CURVE_LEN // 8), 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 3),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        return self.fc(x.view(x.size(0), -1))


class ForwardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 1024), nn.ReLU(),
            nn.Linear(1024, 2048), nn.ReLU(),
            nn.Linear(2048, CURVE_LEN),
        )

    def forward(self, x):
        return self.net(x)


inv_model = InverseNet().to(device)
inv_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "inverse_model.pth"), map_location=device))
inv_model.eval()

fwd_model = ForwardNet().to(device)
fwd_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "forward_model.pth"), map_location=device))
fwd_model.eval()

sx_inv = joblib.load(os.path.join(SAVE_DIR, "sx_inv.pkl"))
sy_inv = joblib.load(os.path.join(SAVE_DIR, "sy_inv.pkl"))
sx_fwd = joblib.load(os.path.join(SAVE_DIR, "sx_fwd.pkl"))
sy_fwd = joblib.load(os.path.join(SAVE_DIR, "sy_fwd.pkl"))

sx_mean = torch.tensor(sx_fwd.mean_, dtype=torch.float32).to(device)
sx_std  = torch.tensor(sx_fwd.scale_, dtype=torch.float32).to(device)
sy_mean = torch.tensor(sy_fwd.mean_, dtype=torch.float32).to(device)
sy_std  = torch.tensor(sy_fwd.scale_, dtype=torch.float32).to(device)


def forward_numpy(params_np):
    p = torch.tensor(params_np, dtype=torch.float32).to(device)
    p_sc = (p - sx_mean) / sx_std
    with torch.no_grad():
        curve = fwd_model(p_sc.unsqueeze(0)).squeeze(0)
    return (curve * sy_std + sy_mean).cpu().numpy()

def mse_numpy(params_np, target_np):
    return float(np.mean((forward_numpy(params_np) - target_np) ** 2))

def adam_solve(target_curve, init_np):
    target = torch.tensor(target_curve, dtype=torch.float32).to(device)
    best_loss, best_params = float("inf"), init_np.copy()

    for _ in range(N_RESTARTS):
        x = torch.tensor(init_np, dtype=torch.float32, requires_grad=True, device=device)
        opt = torch.optim.Adam([x], lr=ADAM_LR)

        for _ in range(ADAM_ITERS):
            # 🚫 NO CLIPPING HERE
            xsc = (x - sx_mean) / sx_std
            pred = fwd_model(xsc.unsqueeze(0)).squeeze(0)
            pred = pred * sy_std + sy_mean
            loss = torch.mean((pred - target) ** 2)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_params = x.detach().cpu().numpy()

    return best_params

def nelder_polish(target_curve, adam_result):

    res = minimize(mse_numpy, adam_result, args=(target_curve,), method="Nelder-Mead")
    return res.x


df = pd.read_excel(TEST_PATH)
freq_cols = [c for c in df.columns if c not in ["L", "W", "G"]]

all_actual, all_pred = [], []


for i, row in df.iterrows():

    print(f"Processing sample {i+1}/{len(df)}")

    mags = np.nan_to_num(pd.to_numeric(row[freq_cols], errors="coerce").values)
    mags = np.pad(mags, (0, CURVE_LEN - len(mags)))[:CURVE_LEN]

    Xin = torch.tensor(sx_inv.transform([mags]), dtype=torch.float32).to(device)

    with torch.no_grad():
        raw = inv_model(Xin)

    # 🚫 NO CLIPPING HERE
    init = sy_inv.inverse_transform(raw.cpu().numpy())[0]

    adam_result = adam_solve(mags, init)
    final = nelder_polish(mags, adam_result)

    actual = row[["L", "W", "G"]].values.astype(float)
    error  = np.abs(actual - final)

    print(f"  Actual    : {actual}")
    print(f"  Predicted (RAW) : {final}")
    print(f"  Error     : {error}")
    print("-"*50)

    all_actual.append(actual)
    all_pred.append(final)


all_actual = np.array(all_actual)
all_pred = np.array(all_pred)

errors = np.abs(all_actual - all_pred)

mae  = errors.mean(axis=0)
rmse = np.sqrt(((all_actual - all_pred) ** 2).mean(axis=0))
mse  = ((all_actual - all_pred) ** 2).mean(axis=0)

print("\nFINAL METRICS (UNCLIPPED):")
print(f"MAE  : {mae}")
print(f"RMSE : {rmse}")
print(f"MSE  : {mse}")


with open(METRICS_FILE, "w") as f:

    f.write("===== TEST METRICS (UNCLIPPED) =====\n")
    f.write(f"{datetime.now()}\n\n")

    f.write("MAE:\n" + str(mae) + "\n\n")
    f.write("RMSE:\n" + str(rmse) + "\n\n")
    f.write("MSE:\n" + str(mse) + "\n\n")

    f.write("===== PER SAMPLE RESULTS =====\n\n")

    for i in range(len(all_actual)):
        actual = all_actual[i]
        pred   = all_pred[i]
        error  = np.abs(actual - pred)

        f.write(f"Sample {i+1}\n")
        f.write(f"  Actual    : {actual}\n")
        f.write(f"  Predicted : {pred}\n")
        f.write(f"  Error     : {error}\n")
        f.write("-"*50 + "\n")

print(f"\n✅ Unclipped results saved to {METRICS_FILE}")

Using device: cpu

Processing sample 1/21
  Actual    : [13.5  3.   5.5]
  Predicted (RAW) : [12.7379875  2.0795262  8.563472 ]
  Error     : [0.76201248 0.92047381 3.06347179]
--------------------------------------------------
Processing sample 2/21
  Actual    : [17.2  6.  10.3]
  Predicted (RAW) : [17.352352   6.1684103  9.882223 ]
  Error     : [0.15235214 0.1684103  0.41777687]
--------------------------------------------------
Processing sample 3/21
  Actual    : [18.5  4.   8. ]
  Predicted (RAW) : [18.241371   3.7646158  8.178711 ]
  Error     : [0.25862885 0.23538423 0.17871094]
--------------------------------------------------
Processing sample 4/21
  Actual    : [19.2  6.  12. ]
  Predicted (RAW) : [19.30716   6.141988 11.517599]
  Error     : [0.10715942 0.1419878  0.48240089]
--------------------------------------------------
Processing sample 5/21
  Actual    : [20.1  3.   6. ]
  Predicted (RAW) : [20.157951  3.14064   5.740466]
  Error     : [0.05795135 0.14064002 0.259

In [6]:
print("\n===== TEST METRICS =====\n")

print(f"{'Metric':<10} {'L':<10} {'W':<10} {'G':<10}")
print("-"*45)

print(f"{'MAE':<10} {mae[0]:<10.3f} {mae[1]:<10.3f} {mae[2]:<10.3f}")
print(f"{'RMSE':<10} {rmse[0]:<10.3f} {rmse[1]:<10.3f} {rmse[2]:<10.3f}")
print(f"{'MSE':<10} {mse[0]:<10.3f} {mse[1]:<10.3f} {mse[2]:<10.3f}")
print(f"{'R2':<10} {r2[0]:<10.3f} {r2[1]:<10.3f} {r2[2]:<10.3f}")


===== TEST METRICS =====

Metric     L          W          G         
---------------------------------------------
MAE        0.851      0.282      1.974     
RMSE       1.484      0.392      3.098     
MSE        2.203      0.154      9.596     
R2         0.966      0.984      0.804     


In [ ]:


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CURVE_LEN   = 1001
SEED        = 42
VAL_SPLIT   = 0.15
DATA_PATH   = r"C:\Users\satya\Desktop\Data\dataset_fullcurve.xlsx"
TEST_PATH   = r"C:\Users\satya\Desktop\Data\testings.xlsx"

# DL Config (intentionally small / few epochs)
DL_EPOCHS   = 20          # ← intentionally low so model doesn't learn properly
DL_BATCH    = 32
DL_LR       = 1e-3

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
print("Loading training data...")
df_train = pd.read_excel(DATA_PATH)
p_cols   = [f"P{i}" for i in range(CURVE_LEN)]

X_all = np.nan_to_num(df_train[p_cols].values.astype(float))
y_all = df_train[["L", "W", "G"]].values.astype(float)

print(f"  Train samples: {len(X_all)}")

print("Loading test data...")
df_test    = pd.read_excel(TEST_PATH)
freq_cols  = [c for c in df_test.columns if c not in ["L", "W", "G"]]
X_test_raw = np.nan_to_num(df_test[freq_cols].values.astype(float))
X_test_raw = np.pad(
    X_test_raw,
    ((0, 0), (0, max(0, CURVE_LEN - X_test_raw.shape[1]))),
    mode="constant"
)[:, :CURVE_LEN]
y_test = df_test[["L", "W", "G"]].values.astype(float)
print(f"  Test samples : {len(X_test_raw)}\n")

# ─────────────────────────────────────────────
# SCALER
# ─────────────────────────────────────────────
scaler_X = StandardScaler().fit(X_all)
scaler_y = StandardScaler().fit(y_all)

X_sc      = scaler_X.transform(X_all)
X_test_sc = scaler_X.transform(X_test_raw)

# ─────────────────────────────────────────────
# METRICS HELPER
# ─────────────────────────────────────────────
def metrics(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    mse  = mean_squared_error(y_true, y_pred, multioutput="raw_values")
    r2   = r2_score(y_true, y_pred, multioutput="raw_values")
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"{'Metric':<10} {'L':<12} {'W':<12} {'G':<12}")
    print(f"{'-'*50}")
    print(f"{'MAE':<10} {mae[0]:<12.3f} {mae[1]:<12.3f} {mae[2]:<12.3f}")
    print(f"{'RMSE':<10} {rmse[0]:<12.3f} {rmse[1]:<12.3f} {rmse[2]:<12.3f}")
    print(f"{'MSE':<10} {mse[0]:<12.3f} {mse[1]:<12.3f} {mse[2]:<12.3f}")
    print(f"{'R2':<10} {r2[0]:<12.3f} {r2[1]:<12.3f} {r2[2]:<12.3f}")
    return {"name": name, "mae": mae, "rmse": rmse, "mse": mse, "r2": r2}

all_results = []

# ─────────────────────────────────────────────
# ML1 — RANDOM FOREST
# ─────────────────────────────────────────────
print("\n[ML1] Random Forest  (n_estimators=50, max_depth=10)")
rf = RandomForestRegressor(
    n_estimators=50,   # intentionally low
    max_depth=10,      # intentionally shallow
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_sc, y_all)
y_pred_rf = rf.predict(X_test_sc)
all_results.append(metrics(y_test, y_pred_rf, "ML1 — Random Forest"))

# ─────────────────────────────────────────────
# ML2 — GRADIENT BOOSTING
# ─────────────────────────────────────────────
print("\n[ML2] Gradient Boosting  (n_estimators=50, max_depth=3)")
gb_base = GradientBoostingRegressor(
    n_estimators=50,   # intentionally low
    max_depth=3,
    learning_rate=0.1,
    random_state=SEED
)
gb = MultiOutputRegressor(gb_base, n_jobs=-1)
gb.fit(X_sc, y_all)
y_pred_gb = gb.predict(X_test_sc)
all_results.append(metrics(y_test, y_pred_gb, "ML2 — Gradient Boosting"))

# ─────────────────────────────────────────────
# ML3 — SVR (Support Vector Regression)
# ─────────────────────────────────────────────
print("\n[ML3] SVR  (RBF kernel, C=1, limited data subsample)")
# SVR is slow on large data — subsample for speed
N_SVR = min(3000, len(X_sc))
idx   = np.random.choice(len(X_sc), N_SVR, replace=False)
svr_base = SVR(kernel="rbf", C=1.0, epsilon=0.1)
svr      = MultiOutputRegressor(svr_base, n_jobs=-1)
svr.fit(X_sc[idx], y_all[idx])
y_pred_svr = svr.predict(X_test_sc)
all_results.append(metrics(y_test, y_pred_svr, "ML3 — SVR"))

# ─────────────────────────────────────────────
# ML4 — KNN REGRESSOR
# ─────────────────────────────────────────────
print("\n[ML4] KNN  (k=10)")
knn = KNeighborsRegressor(n_neighbors=10, n_jobs=-1)
knn.fit(X_sc, y_all)
y_pred_knn = knn.predict(X_test_sc)
all_results.append(metrics(y_test, y_pred_knn, "ML4 — KNN Regressor"))

# ─────────────────────────────────────────────
# ML5 — RIDGE REGRESSION (linear baseline)
# ─────────────────────────────────────────────
print("\n[ML5] Ridge Regression  (alpha=1.0)")
ridge = Ridge(alpha=1.0)
ridge.fit(X_sc, y_all)
y_pred_ridge = ridge.predict(X_test_sc)
all_results.append(metrics(y_test, y_pred_ridge, "ML5 — Ridge Regression"))

# ─────────────────────────────────────────────
# DL1 — SIMPLE 1D-CNN  (intentionally weak)
# ─────────────────────────────────────────────
print(f"\n[DL1] Simple 1D-CNN  ({DL_EPOCHS} epochs — intentionally undertrained)")

class SimpleCNN(nn.Module):
    """Shallow 1D-CNN, intentionally simple to show weakness vs InverseNet."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(4),                              # aggressive pool
            nn.Conv1d(16, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(4),
        )
        flat = 32 * (CURVE_LEN // 16)
        self.fc = nn.Sequential(
            nn.Linear(flat, 64), nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        return self.fc(x.view(x.size(0), -1))

# Scale for DL
scaler_X_dl = StandardScaler().fit(X_all)
scaler_y_dl = StandardScaler().fit(y_all)

X_tr_dl = torch.tensor(scaler_X_dl.transform(X_all), dtype=torch.float32)
y_tr_dl = torch.tensor(scaler_y_dl.transform(y_all), dtype=torch.float32)
X_ts_dl = torch.tensor(scaler_X_dl.transform(X_test_raw), dtype=torch.float32)

dl_loader = DataLoader(TensorDataset(X_tr_dl, y_tr_dl), batch_size=DL_BATCH, shuffle=True)

cnn = SimpleCNN().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=DL_LR)
loss_fn = nn.MSELoss()

for ep in range(1, DL_EPOCHS + 1):
    cnn.train()
    total = 0.0
    for xb, yb in dl_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = cnn(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    avg = total / len(dl_loader)
    print(f"  Epoch {ep:>2}/{DL_EPOCHS}  Loss: {avg:.5f}")

cnn.eval()
with torch.no_grad():
    y_pred_cnn_sc = cnn(X_ts_dl.to(device)).cpu().numpy()
y_pred_cnn = scaler_y_dl.inverse_transform(y_pred_cnn_sc)
all_results.append(metrics(y_test, y_pred_cnn, "DL1 — Simple 1D-CNN (undertrained)"))

# ─────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("  SUMMARY — ALL BASELINE MODELS  (compare with Hybrid CNN results)")
print("="*70)
print(f"{'Model':<35} {'MAE_L':>7} {'MAE_W':>7} {'MAE_G':>7}  {'R2_L':>7} {'R2_W':>7} {'R2_G':>7}")
print("-"*70)
for r in all_results:
    print(f"{r['name']:<35} "
          f"{r['mae'][0]:>7.3f} {r['mae'][1]:>7.3f} {r['mae'][2]:>7.3f}  "
          f"{r['r2'][0]:>7.3f} {r['r2'][1]:>7.3f} {r['r2'][2]:>7.3f}")
print("="*70)


Device: cpu

Loading training data...
  Train samples: 2789
Loading test data...
  Test samples : 21


[ML1] Random Forest  (n_estimators=50, max_depth=10)

  ML1 — Random Forest
Metric     L            W            G           
--------------------------------------------------
MAE        1.616        1.069        4.407       
RMSE       2.843        1.867        6.922       
MSE        8.085        3.484        47.920      
R2         0.874        0.644        0.022       

[ML2] Gradient Boosting  (n_estimators=50, max_depth=3)

  ML2 — Gradient Boosting
Metric     L            W            G           
--------------------------------------------------
MAE        1.384        1.495        5.341       
RMSE       2.397        2.289        7.857       
MSE        5.746        5.241        61.728      
R2         0.910        0.464        -0.259      

[ML3] SVR  (RBF kernel, C=1, limited data subsample)

  ML3 — SVR
Metric     L            W            G           
------------------